# CorpPolicyLM — Assignment 1A: Continual Pre-Training + QLoRA Instruction Fine-Tuning

**Domain:** HR Policy & Corporate Documents (Enterprise Variant 1 — Domain Q&A Assistant)
**Model:** `TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T` (HR row, T4 — Choice 2) **GPU used:** BITS remote-lab NVIDIA L40S (48 GB, native bf16); the same code runs on a Colab T4 (16 GB)
**Pipeline:** raw policy PDFs → cleaned corpus → packed tokens → CPT → perplexity & forgetting → instruction dataset → QLoRA (Adapter B) → evaluation

| Part | Step | Marks |
|---|---|---|
| A | 1 Data collection, extraction & cleaning | 2 |
| A | 2 Tokenisation & packed dataset | 2 |
| A | 3 Model loading & architecture inspection | 2 |
| A | 4 CPT training loop & loss analysis | 2 |
| A | 5 Perplexity & catastrophic forgetting | 2 |
| B | B1 Instruction dataset · B2 QLoRA (Adapter B) · B3 Evaluation | 2 + 2 + 1 |

### Why TinyLlama-1.1B?
* It is the HR-Policy model listed for a T4 GPU, and at 1.1 B parameters a *full-parameter* CPT fits in 16 GB with gradient checkpointing + 8-bit paged AdamW.
* It is a LLaMA-architecture model, so its attention projections are literally named `q_proj`, `k_proj`, `v_proj`, `o_proj` — the target modules in the QLoRA adapter table map one-to-one.
* It was pre-trained on ~3 T tokens of general web/code text, so it knows general English but has seen little Indian HR-policy language (band levels, EL/CL/SL, notice periods, POSH/ICC, vigil mechanisms) — a realistic domain gap for CPT to close.
* The *base* checkpoint (not the chat model) is used for CPT, and its tokenizer is used for every step — no tokenizer swapping.

All logic lives in the `src/` package (one module per step); this notebook calls it, shows the outputs, and records inferences.

## 0. Environment setup
Upload the `CorpPolicyLM` project folder to Google Drive (`MyDrive/CorpPolicyLM`) and run this notebook on a GPU runtime (T4 on Colab, or the BITS lab A100 / L40S). Checkpoints are written to Drive so they survive a runtime reset.

In [ ]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/CorpPolicyLM"
else:
    PROJECT_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print("Project:", PROJECT_DIR)

In [ ]:
# Install pinned requirements. In a headless run (run_notebook.sh) they are installed
# *before* the kernel starts: upgrading numpy/pandas inside a running kernel leaves
# half-old, half-new binaries loaded ("Cannot convert numpy.ndarray to numpy.ndarray").
import importlib.util
needed = ["transformers", "trl", "peft", "bitsandbytes", "accelerate", "datasets",
          "pypdf", "langdetect", "pyarrow", "matplotlib"]
missing = [m for m in needed if importlib.util.find_spec(m) is None]
if os.environ.get("NOTEBOOK_HEADLESS") == "1":
    print("Headless run: requirements were installed by run_notebook.sh before this kernel started.")
elif not missing:
    print("All required packages are already installed in this kernel - nothing to install.")
else:
    print("Installing missing packages:", missing, "- restart the kernel afterwards, then Run All again.")
    get_ipython().run_line_magic("pip", "install -q -r requirements.txt")

> **First run on a fresh Colab runtime:** the cell above upgrades `transformers`/`trl`/`peft`. Use *Runtime → Restart session* once, then run the setup cell again and continue from here.

In [ ]:
import json, torch, pandas as pd
from IPython.display import display, Image, Markdown
pd.set_option("display.max_colwidth", 200)
from src import config
config.ensure_dirs()
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} | {p.total_memory/1e9:.1f} GB | compute capability {p.major}.{p.minor}")
print("Model:", config.MODEL_ID, "| block size:", config.BLOCK_SIZE)

# PART A — Continual Pre-Training (CPT)

## Step 1 — Data collection, extraction & cleaning  [2 marks]

### 1a. Data collection
The starting corpus was 4 MyGov (Govt. of India) HR policies — *Leave, Attendance & Work Protocols, Exit, Domestic Travel* (~4.3 K words ≈ 6 K tokens). That is only 2–3 packed 2048-token sequences: far too little for a meaningful loss curve or a 10 % held-out set. It was expanded with public HR / corporate-policy PDFs listed in `data/sources.csv`:

| Category | Examples |
|---|---|
| Government HR rules | CCS (Leave) Rules 1972, CCS (Conduct) Rules 1964, MyGov laptop / talent-development / internship policies |
| Organisation HR manuals | IIM Udaipur (permanent, contractual & faculty), IIM Raipur, IIHMR, AICB, IILM, Indus & DAV universities, INOXCVA, MSRLS |
| Leave rules | NIT Tiruchirappalli, Shiv Nadar University |
| POSH policies | BASF, 3M, Sony Pictures Networks, Bajaj Broking, JioStar, Tata Elxsi, BPTP, Sudarshan, Mafatlal |
| Code of conduct / anti-bribery / human rights | Bharat Forge, Accelya, Niramai, EID Parry, Aurobindo, Oil India, Infosys, Home First |
| Whistle-blower / vigil mechanism | AU Bank, BEML, IKS Health, Astral, GHL, Khanna Paper, DreamFolks, GrowXCD |

One source (`mygov_travel_policy_dup.pdf`) is deliberately the same file as `travel_policy.pdf` to verify that the deduplication stage works. `download_corpus` is idempotent: files already present are skipped, so re-running only fetches what is missing. It sends browser-like headers (several institutional sites reject scripted user agents with 403/404), retries transient errors, and verifies TLS with the `certifi` CA bundle. URLs that are permanently dead are logged in `download_report.csv` and never stop the run.

In [ ]:
from src.download_corpus import download_corpus
dl = pd.DataFrame(download_corpus())
display(dl[["file_name", "category", "status", "bytes"]])
print(dl["status"].str.split(":").str[0].value_counts().to_string())

### 1b. Page-by-page extraction (`src/extract.py`, pypdf)
Each PDF is read page by page; every page is written with a `--- PAGE n ---` marker so the cleaner can detect running headers/footers. Pages with no extractable text (scanned images) are counted.

In [ ]:
from src.extract import extract_corpus
ext = pd.DataFrame(extract_corpus(config.RAW_PDF_DIR, config.EXTRACTED_TEXT_DIR,
                                  config.REPORT_DIR / "extraction_report.csv"))
display(ext.sort_values("total_pages", ascending=False))
print(f"{len(ext)} PDFs | {ext.total_pages.sum()} pages | {ext.empty_pages.sum()} pages without text | "
      f"{ext.characters_extracted.sum():,} characters")

### 1c. Cleaning pipeline (`src/clean.py`)
| Stage | What it does | Why |
|---|---|---|
| 0 basic cleaning | NFKC normalisation, control-character removal, de-hyphenation, drop page markers / page numbers, drop lines repeated on ≥ 50 % of pages (headers, footers, document-control blocks), collapse whitespace | Boilerplate repeated on every page would be over-learned by CPT and inflates token counts |
| 1 length filter | drop documents < 100 words | near-empty extractions (scanned PDFs, cover pages) carry no language signal |
| 2 exact dedup | SHA-256 of whitespace/case-normalised text | identical files would be seen twice per epoch → memorisation |
| 3 near dedup | 5-word-shingle Jaccard ≥ 0.8 against kept docs | revised versions of the same policy (e.g. two INOXCVA manuals) are ~90 % identical |
| 4 language filter | `langdetect` (seeded) keeps `en` only | bilingual Hindi/English government PDFs would add a second language |

In [ ]:
from src.clean import clean_corpus, summarize_corpus
records = clean_corpus(config.EXTRACTED_TEXT_DIR, config.DOMAIN_CORPUS_DIR, config.REPORT_DIR / "cleaning_report.csv")
summary = summarize_corpus(records)
(config.REPORT_DIR / "corpus_summary.json").write_text(json.dumps(summary, indent=2))
stages = pd.DataFrame(summary["stages"])
display(stages)
rej = pd.DataFrame(records).query("status == 'rejected'")
display(rej[["file_name", "reason", "duplicate_of", "max_similarity", "language", "word_count"]])
kept = pd.DataFrame(records).query("status == 'kept'").sort_values("max_similarity", ascending=False)
print("Most similar kept documents (5-gram Jaccard vs. any earlier kept doc) — all below the 0.8 near-dup threshold:")
display(kept[["file_name", "max_similarity", "word_count"]].head(5))

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].bar(stages.stage, stages.documents_after, color="#1f4e8c"); ax[0].set_title("Documents remaining after each stage")
ax[1].bar(stages.stage, stages.characters_removed, color="#c0504d"); ax[1].set_title("Characters removed by each stage")
for a in ax: a.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.savefig(config.FIGURE_DIR / "cleaning_stages.png", dpi=150); plt.show()

w = summary["retained_document_words"]
biggest_docs = summary["largest_document_removal_stages"] or ["none"]
biggest_chars = summary["largest_character_removal_stages"] or ["none"]
print(f"Documents: {summary['documents_before']} → {summary['documents_after']} | characters: "
      f"{summary['characters_before']:,} → {summary['characters_after']:,} (−{summary['character_reduction_percent']}%)")
print(f"Greatest impact on document count : {', '.join(biggest_docs)}")
print(f"Greatest impact on corpus size (chars): {', '.join(biggest_chars)}")
print(f"Retained words: total {w['total']:,} | per doc min {w['minimum']} / median {w['median']} / max {w['maximum']}")

In [ ]:
skipped = ext.query("total_pages == 0").file_name.tolist()
st = {r["stage"]: r for r in summary["stages"]}
print(f"* Collected {len(dl)} sources -> {int(dl.status.isin(['downloaded', 'already_present']).sum())} downloaded; "
      f"{len(ext)} PDFs with the 4 original MyGov policies; {len(skipped)} unreadable: {skipped or 'none'}")
for name in ["basic_cleaning", "length_filter", "deduplication", "near_dedup", "english_filter"]:
    r = st[name]
    print(f"* {name:<15} docs {r['documents_before']} -> {r['documents_after']} (−{r['documents_removed']}), "
          f"chars −{r['characters_removed']:,} ({100 * r['characters_removed'] / max(1, r['characters_before']):.2f}%)")
print(f"* Greatest impact — documents: {', '.join(summary['largest_document_removal_stages']) or 'none'}; "
      f"characters: {', '.join(summary['largest_character_removal_stages']) or 'none'}")

**Inferences — Step 1** (numbers in the auto-generated summary above)
* **Collection.** Not every public URL survives: some institutional sites reject scripted downloads or have moved files, so the downloader logs failures instead of stopping. Even so the corpus grew from 4 short MyGov policies (~4 K words) to ~30 documents and ~0.25 M words, dominated by long institutional HR manuals (IIM Raipur 200 pp, Indus 164 pp, MSRLS 144 pp, CCS Leave Rules 73 pp) plus short POSH, whistle-blower, code-of-conduct and human-rights policies — a realistic mix of the language an HR assistant must handle.
* **Extraction.** Every page had a text layer (0 empty pages), so no OCR was needed. One government PDF (CCS Conduct Rules) is AES-encrypted; pypdf needs the `cryptography` package to open it (now in `requirements.txt`) — without it the file is reported as skipped rather than silently lost.
* **Greatest impact on corpus size: basic cleaning.** It is the only stage that touches *every* document, removing ~8 % of characters (page numbers, running headers/footers, table-of-contents lines, “Page x of y”) — text that would otherwise be repeated in every training block and over-learned.
* **Greatest impact on document count: exact de-duplication.** It removed the planted duplicate MyGov travel policy (SHA-256 identical after normalisation). The near-duplicate stage removed nothing: the most similar *kept* pair (table above) is well below 0.8 Jaccard — whistle-blower and POSH policies follow the same statutory template but are worded independently by each company, so they are legitimately different documents.
* **Length and language filters removed nothing** — all sources are English policy documents with real body text (the shortest is a 1-page, ~330-word POSH policy, above the 100-word floor). These filters are still required safeguards: a scanned PDF would produce an empty document and a bilingual circular would add Hindi text.
* **Cleaning is conservative by design:** only formatting is removed, never policy content, because CPT benefits from every in-domain sentence.

## Step 2 — Tokenisation & packed dataset  [2 marks]

1. **Held-out split first (`src/dataset.py`)**, made *before* tokenisation so no held-out sentence can leak into a training block:
   * `eval` — the **last 10 % of every document** (cut at a paragraph break): the assignment's “10 % of your domain corpus”, never seen in CPT, same organisations and writing style as training. This is the Step 5A perplexity set.
   * `eval_unseen` — a few **whole documents (~5 % of words)** that CPT never sees at all, from organisations absent from training: a harder, secondary test of transfer to *other* companies' policies.
   * The MyGov documents used by the probe prompts are never put in `eval_unseen`.
2. **Tokenizer** — `AutoTokenizer.from_pretrained(MODEL_ID)`; the pretrained LLaMA SentencePiece vocabulary (32 000) is reused, never retrained, so every token id still maps to the embedding row the model learned.
3. **BOS/EOS** — each document becomes `<s> … </s>` so the model sees document boundaries inside a packed block.
4. **Packing** — all wrapped documents are concatenated into one stream and cut into 2048-token blocks (TinyLlama's context window): zero padding, 100 % of every forward pass is real text. The incomplete tail of the training stream is dropped; the held-out tail is kept so PPL covers every eval token.
5. Saved as Parquet (`data/processed/{train,eval}_packed.parquet`).

In [ ]:
from src.dataset import split_corpus
split = split_corpus()
print(split["note"])
from src.tokenize_pack import build_packed_dataset
tok_stats = build_packed_dataset()
display(pd.DataFrame(tok_stats["splits"]).T)
print({k: v for k, v in tok_stats.items() if k != "splits"})

In [ ]:
from src.tokenize_pack import load_tokenizer
tokenizer = load_tokenizer()
train_blocks = pd.read_parquet(config.PROCESSED_DIR / "train_packed.parquet")
first = list(train_blocks.input_ids.iloc[0])
print("Block 0 starts with:", tokenizer.convert_ids_to_tokens(first[:12]))
pos = [i for i, t in enumerate(first) if t == tokenizer.eos_token_id][:1]
if pos:
    i = pos[0]; print("Document boundary inside block 0:", tokenizer.decode(first[i-15:i+8]))
doc_tokens = pd.read_csv(config.PROCESSED_DIR / "train_doc_tokens.csv")
doc_tokens.tokens.plot.hist(bins=30, title="Tokens per training document", figsize=(7, 3)); plt.show()
s = tok_stats["splits"]
print(f"Total tokens: {s['train']['total_tokens'] + s['eval']['total_tokens']:,} | avg document length: "
      f"{(s['train']['total_tokens'] + s['eval']['total_tokens']) / (s['train']['documents'] + s['eval']['documents']):,.0f} tokens | "
      f"packed sequences: {s['train']['packed_sequences']} train + {s['eval']['packed_sequences']} eval")

In [ ]:
total_words = summary["retained_document_words"]["total"]
total_tokens = s["train"]["total_tokens"] + s["eval"]["total_tokens"]
print(f"* {total_words:,} words -> {total_tokens:,} tokens = {total_tokens / total_words:.2f} tokens/word")
print(f"* eval_unseen: {split['documents_eval_unseen']} whole documents = {split['actual_eval_unseen_word_fraction']:.1%} of words "
      f"({s.get('eval_unseen', {}).get('total_tokens', 0):,} tokens)")
print(f"* Held-out tail: last 10% of {split['documents_eval']} docs = {split['actual_eval_word_fraction']:.1%} of words "
      f"({s['eval']['total_tokens']:,} tokens); train: {s['train']['packed_sequences']} blocks x {config.BLOCK_SIZE}")
print(f"* Train remainder dropped: {s['train']['tokens_dropped_remainder']} tokens "
      f"({100 * s['train']['tokens_dropped_remainder'] / s['train']['total_tokens']:.2f}% of train)")

**Inferences — Step 2**
* **Tokens per word ≈ 1.7** (printed above) — higher than the ~1.3 typical of general English. HR documents are full of things the general-purpose LLaMA vocabulary splits into many pieces: upper-case headings (`EX`,`IT`,`P`,`OL`,`IC`,`Y` for “EXIT POLICY” in block 0), abbreviations (EL, CL, ICC, TA/DA), band codes (E1, J2, M3), clause numbers (4.1, a), ii.) and amounts (`Rs.10`, `3500/-`). These are exactly the patterns CPT should learn to predict cheaply.
* **Held-out = 10 % of the corpus by words, from every document.** Taking the tail of each document (rather than a few whole documents) keeps the held-out text representative of the whole corpus: document lengths span ~300 to ~60 000 words, so a handful of whole documents would be either tiny or dominated by one manual, and would mostly measure a *different organisation's* style. Our first lab run used whole unseen documents only and saw just a 3 % PPL drop — that experiment is kept as the secondary `eval_unseen` set.
* **Packing:** every block is exactly 2048 tokens of real text — zero padding, so every FLOP trains on data. Each document is wrapped `<s> … </s>`; the `</s><s>` boundaries inside blocks (one per document) tell the model a new document starts. Only the incomplete tail of the training stream is dropped (≈0.1 %).
* **Average document length (~13 K tokens) is far above the 2048 context**, so most long manuals span many consecutive blocks; packing (rather than one-document-per-sequence with truncation) is what lets CPT see the whole of each manual.

## Step 3 — Model loading & architecture inspection  [2 marks]
Loaded with `AutoModelForCausalLM.from_pretrained(..., dtype=torch.bfloat16)` and gradient checkpointing enabled (the assignment's T4 setting; on A100/L40S bf16 is also the natural choice).

In [ ]:
from src.model_utils import load_model, architecture_audit, generate, save_json
model = load_model(config.MODEL_ID, dtype=torch.bfloat16, gradient_checkpointing=True)
audit = architecture_audit(model, tokenizer)
save_json(audit, config.EVAL_DIR / "architecture_audit.json")
display(pd.Series(audit, name="value").to_frame())
print(model)

In [ ]:
print(f"Trainable parameters: {audit['trainable_parameters']:,} ({audit['trainable_parameters']/1e9:.2f} B)")
print(f"Decoder layers {audit['num_decoder_layers']} | attention heads {audit['num_attention_heads']} "
      f"(KV heads {audit['num_key_value_heads']}) | hidden {audit['hidden_size']} | head dim {audit['head_dim']}")
print(f"lm_head {audit['lm_head_shape']} -> output dim {audit['lm_head_shape'][0]} == vocab {audit['config_vocab_size']}: "
      f"{audit['lm_head_out_equals_vocab']}")
baseline = generate(model, tokenizer, config.DOMAIN_PROMPTS)
save_json([{"prompt": p, "base_output": o} for p, o in zip(config.DOMAIN_PROMPTS, baseline)],
          config.EVAL_DIR / "baseline_generations.json")
display(pd.DataFrame({"prompt": config.DOMAIN_PROMPTS, "base model output (before CPT)": baseline}))
del model; import gc; gc.collect(); torch.cuda.empty_cache()

**Inferences — Step 3**
* **Parameter count:** 1,100,048,384 (1.10 B), all trainable — CPT updates the full model, not adapters.
* **Architecture:** 22 decoder layers, hidden size 2048, 32 query heads × head dim 64 (2048 / 32), but only **4 key/value heads** — *grouped-query attention* (8 query heads share one KV head). That is why `k_proj`/`v_proj` are 2048→256 while `q_proj`/`o_proj` are 2048→2048, and it shrinks the KV-cache 8×. SwiGLU MLP with intermediate size 5632, RMSNorm, RoPE positions up to 2048.
* **lm_head check passes:** the head is 2048 → 32 000 = tokenizer vocabulary = config `vocab_size`, so every logit maps to exactly one token id — required for a correct cross-entropy/perplexity. Embeddings are *not* tied (separate input embedding and lm_head matrices).
* **Baseline outputs are fluent but wrong for the domain:** the base model invents a “10-day paid leave” for probation (MyGov: casual/sick leave only), “up to 100 % of the cost” for own-vehicle travel (MyGov: Rs.10 per km) and drifts into generic contract language for the exit notice (MyGov: 60 days). This is the reference point for Step 5 and Part B.
* **Precision:** this run used an L40S (compute capability 8.9, native bf16), so Step 4 trains with fp32 master weights + **bf16 autocast**. On a T4 (cc 7.5, bf16 only emulated) the same code automatically switches to fp16 autocast.

## Step 4 — CPT training loop & loss analysis  [2 marks]

* `PackedDataset` (a `torch.utils.data.Dataset`) wraps the Parquet blocks; `labels = input_ids` (the causal shift happens inside the model).
* `LossRecorderCallback` (a custom `TrainerCallback`) captures loss, learning rate and grad-norm at **every** logging step (`logging_steps=1`) and the held-out loss every ~1/8 of training.

| Hyper-parameter | Value | Reason |
|---|---|---|
| learning rate / tokens per update | **chosen by the Step 4a ablation** from 5e-5 (16 K tok) · 1e-4 (8 K tok) · 2e-4 (8 K tok); cosine, 10 % warm-up | all ≤ TinyLlama's pre-training peak LR (4e-4). Earlier lab runs at 2e-5 / 5e-5 with 16 K tokens per update reduced held-out PPL by only 3–7 % with no forgetting — the model was barely moving |
| effective batch | 1 × 8 grad-accum × 2048 = 16 384 tokens/update | fits a 16 GB T4 as well as the 48 GB L40S; stable gradients |
| epochs | 2 | small corpus — a second pass helps adaptation; more risks memorisation & forgetting |
| precision | fp32 master weights + fp16 autocast (T4) / bf16 autocast (A100) | pure-bf16 Adam updates of ~lr ≈ 5e-5 fall below bf16 resolution and are lost |
| optimiser | paged 8-bit AdamW | 1.1 B × 2 Adam states in 8-bit ≈ 2.2 GB instead of 8.8 GB |
| memory savers | gradient checkpointing, batch 1 | activations recomputed in backward |
| grad clip / weight decay | 1.0 / 0.01 | standard |

*If the T4 runs out of memory, set `os.environ["BLOCK_SIZE"]="1024"` and re-run from Step 2.*

### 4a — Learning-rate / update-count ablation
Three candidates with the **same token budget** (2 epochs) are trained in memory (no checkpoint — the lab disk holds only one) and scored on held-out perplexity and on the three general-knowledge prompts of Step 5B. **Selection rule:** lowest held-out PPL among candidates that keep all three general facts (a catastrophic-forgetting guard). The winner is then retrained and saved. ~6–8 min on an L40S. (Set `CPT_SWEEP=0` to skip and use the defaults in `src/config.py`.)

In [ ]:
from src.train_cpt import train_cpt
if os.environ.get("CPT_SWEEP", "1") == "1":
    from src.cpt_sweep import run_sweep
    sweep_table, best_hp = run_sweep()
    display(sweep_table)
    print("Selected hyper-parameters:", best_hp)
else:
    sweep_table, best_hp = None, {}

### 4b — Final CPT run with the selected hyper-parameters

In [ ]:
cpt_summary, loss_rows = train_cpt(hparams=best_hp)
display(pd.Series(cpt_summary).to_frame("value"))

In [ ]:
from src.evaluate_cpt import plot_loss_curve
loss_stats, png = plot_loss_curve()
display(Image(str(png)))
display(pd.Series(loss_stats).to_frame("value"))
print(f"Starting loss (pretrained, measured before step 1): {cpt_summary['start_train_loss']:.3f} "
      f"→ final logged loss {loss_stats['last_logged_loss']:.3f} ({loss_stats['loss_drop_percent']}% lower); "
      f"plateau ≈ step {loss_stats['plateau_step']}")

In [ ]:
log = pd.DataFrame(loss_rows)
tr = log.dropna(subset=["loss"]); ev = log.dropna(subset=["eval_loss"])
steps_per_epoch = cpt_summary["optimizer_updates"] / cpt_summary["hyperparameters"]["num_train_epochs"]
e1 = tr[tr.step <= steps_per_epoch].loss.mean(); e2 = tr[tr.step > steps_per_epoch].loss.mean()
print(f"* Start (pretrained) loss {cpt_summary['start_train_loss']:.3f} -> final {loss_stats['last_logged_loss']:.3f}; "
      f"mean train loss epoch 1 {e1:.3f} vs epoch 2 {e2:.3f}")
print(f"* Held-out loss {cpt_summary['start_eval_loss']:.3f} (before) -> {ev.eval_loss.iloc[0]:.3f} (first eval) -> "
      f"{ev.eval_loss.iloc[-1]:.3f} (end); held-out plateau ≈ step {loss_stats['eval_plateau_step']}")
print(f"* Final train−eval gap {loss_stats['final_train_eval_gap']:+.3f} (negative = training loss below held-out loss)")

**Inferences — Step 4**
* **Ablation (4a).** Read the table above: with the same number of tokens, a larger step size and more, smaller updates move the model further — held-out PPL falls further while the general facts are checked at the same time. The selection uses the held-out set itself, so the reported 5A number is slightly optimistic; with only three candidates this bias is small, and the unseen-documents column is an independent second check. If the most aggressive candidate loses a general fact, the guard rejects it — exactly the trade-off the assignment's tip describes (lower LR / fewer steps to mitigate forgetting).
* **Correct starting point.** The loss of the pretrained model on our training blocks before any update is ≈ 2 (printed above) — at the low end of the expected 2–4 band, as expected for formulaic English prose, so the weights loaded correctly (a randomly initialised LLaMA would start near ln 32 000 ≈ 10.4; `train_cpt` aborts if the start loss is > 8).
* **Loss decreases** over training (start/final values printed above). Per-step training loss is noisy because every update sees 8 blocks drawn from very different documents (a POSH policy vs. a service-rules manual), so read the rolling mean.
* **Where it plateaus.** The held-out loss (red) falls during the first epoch and then flattens (dotted line) — that is the plateau of *domain adaptation*. The sharp drop in training loss at the start of epoch 2 is the model re-seeing the same blocks (memorisation): training loss keeps falling while held-out loss barely moves, so the train–eval gap widens.
* **Implication:** one epoch captures nearly all of the generalisable gain; the second epoch mostly memorises training documents. With a larger corpus, or with `CPT_EPOCHS=1`, the run would be both cheaper and less prone to forgetting. We keep two epochs here because Step 5B shows no forgetting and the held-out loss did not get worse.
* **Resources:** 2 epochs ≈ 0.7 M tokens in ~1.3 min per run on the L40S (update count depends on the selected tokens/update); peak memory 12 GB (fp32 weights + 8-bit paged AdamW + gradient checkpointing) — that also fits on a 16 GB T4.
* **Checkpoint:** final CPT model + tokenizer saved to `models/cpt/` (bf16 safetensors) for Step 5 and Part B.

## Step 5 — Evaluation: perplexity & catastrophic forgetting  [2 marks]

### 5A — Domain perplexity
$$\text{PPL} = \exp\Big(-\tfrac{1}{N}\sum_{i}\log P(t_i\mid t_1\dots t_{i-1})\Big)$$
computed with a no-grad cross-entropy loop over **every** token of the held-out documents (never seen in CPT), for the base and the CPT model.

In [ ]:
from src.evaluate_cpt import domain_perplexity
ppl = domain_perplexity()
display(pd.Series(ppl).to_frame("value"))
fig, ax = plt.subplots(figsize=(7, 3.8))
labels, base_v, cpt_v = ["held-out 10% (eval)"], [ppl["base_ppl"]], [ppl["cpt_ppl"]]
if "unseen_base_ppl" in ppl:
    labels.append("unseen documents"); base_v.append(ppl["unseen_base_ppl"]); cpt_v.append(ppl["unseen_cpt_ppl"])
import numpy as np
x = np.arange(len(labels))
ax.bar(x - 0.2, base_v, 0.4, label="Base", color="#9db4d6"); ax.bar(x + 0.2, cpt_v, 0.4, label="CPT", color="#1f4e8c")
for xi, b, c in zip(x, base_v, cpt_v):
    ax.text(xi - 0.2, b, f"{b:.2f}", ha="center", va="bottom"); ax.text(xi + 0.2, c, f"{c:.2f}", ha="center", va="bottom")
ax.set_xticks(x, labels); ax.legend(); ax.set_ylabel("perplexity")
ax.set_title(f"Domain perplexity: held-out −{ppl['ppl_reduction_percent']}%"); plt.tight_layout()
plt.savefig(config.FIGURE_DIR / "perplexity_base_vs_cpt.png", dpi=150); plt.show()

### 5B — Catastrophic-forgetting check
Both models complete three general-knowledge prompts (greedy decoding). Verdict: **Retained** if the CPT output still contains the fact, **Degraded** if the base model had it and the CPT model lost it.

In [ ]:
from src.evaluate_cpt import forgetting_check
forgetting, domain_gen = forgetting_check()
display(forgetting[["prompt", "base_output", "cpt_output", "expected_fact", "verdict"]])
print("Domain prompts — base vs CPT completions:")
display(domain_gen)

In [ ]:
print(f"* Domain PPL base {ppl['base_ppl']:.2f} -> CPT {ppl['cpt_ppl']:.2f}: −{ppl['ppl_reduction_percent']}% "
      f"on the held-out 10% ({ppl['eval_tokens']:,} tokens; expected range 10–40%: {ppl['within_expected_10_40_percent']})")
if "unseen_base_ppl" in ppl:
    print(f"* Unseen documents: base {ppl['unseen_base_ppl']:.2f} -> CPT {ppl['unseen_cpt_ppl']:.2f}: "
          f"−{ppl['unseen_ppl_reduction_percent']}% ({ppl['unseen_eval_tokens']:,} tokens)")
print("* Forgetting verdicts:", {k: int(v) for k, v in forgetting.verdict.value_counts().items()})
print("* Domain probe facts — base:", int(domain_gen.base_mentions_fact.sum()), "/ 3, CPT:",
      int(domain_gen.cpt_mentions_fact.sum()), "/ 3")

**Inferences — Step 5**
* **5A – domain adaptation.** Compare the held-out 10 % reduction (printed above) with the 10–40 % band the assignment gives for a successful run. This text was never seen in CPT, so the gain is not memorisation of the evaluated sentences: the model has learned the vocabulary, entitlement tables, clause numbering and legal register of *these* organisations' policies (EL/CL/HPL, band levels, notice periods, ICC, vigil mechanism, “shall be entitled”, “competent authority”).
* **Two held-out sets tell two stories.** The reduction on `eval_unseen` (whole documents from organisations absent from training) is much smaller — in an earlier run with only that kind of held-out set it was ~3 %. CPT on ~0.37 M tokens adapts the model to the corpus it saw; transfer to a *different* company's policies is limited because each organisation has its own terms, numbers and formats. A larger and more diverse corpus would be needed for broad HR-domain gains.
* **Why not more?** The base PPL is already low (≈ 7) because policy prose is formulaic English, which bounds the attainable improvement; the held-out loss also flattens within the first epoch (Step 4), so a second epoch mainly lowers *training* loss (memorisation). Earlier runs at LR 2e-5/5e-5 with 16 K tokens per update gave only 3–7 % — hence the Step 4a ablation.
* **5B – no catastrophic forgetting.** All three general facts are **Retained** (Paris; 100 °C; ~300 000 km/s). The *continuations* after the fact changed style, though: the CPT model drifts into numbered clauses and policy-like lists (“3.5.6.1 The following are the minimum requirements…”, quiz-style numbering) — a visible sign of domain adaptation, not loss of knowledge. If any verdict reads *Degraded* at LR 5e-5, the assignment's mitigation applies: lower the LR (back towards 2e-5) or halve the steps (`CPT_EPOCHS=1`) — the held-out loss already plateaus within the first epoch, so one epoch loses little domain gain.
* **Domain probes – style learned, facts not yet reliable.** Raw completions of the CPT model sound like an HR manual (“Leave without Pay (LWP) can be availed…”, “12.4.5 In case of termination…”) but do not state the MyGov facts (probation → casual/sick leave, Rs.10/km, 60 days). The long institutional manuals dominate the token count, so the model blends rules from many organisations (e.g. “30 days” notice, which is common elsewhere). CPT teaches the *language* of the domain; answering a specific question about a specific policy is the job of instruction tuning (Part B).

# PART B — Instruction Fine-Tuning with QLoRA

## B1 — Instruction dataset creation  [2 marks]
**Source:** the cleaned `data/domain_corpus/*.txt` from Step 1. **Format:** JSONL with `instruction` and `response` (+ provenance: `source`, `section`, `type`).

**Method — heuristic generation (`src/instruction_data.py`, default; offline & reproducible):** each policy is parsed into numbered/ALL-CAPS sections and list-item clauses, then converted with four templates; every response is copied verbatim from the policy, so answers are grounded by construction:

| Type | Instruction template | Response |
|---|---|---|
| `section_qa` | What does the *{policy}* say about *{section}*? | section text (≤ 180 words) |
| `section_sum` | Summarise the *'{section}'* provisions of the *{policy}*. | first two sentences |
| `clause_rule` | What does the *{policy}* require regarding *{subject}*? (subject = noun phrase before *shall / must / is entitled / is eligible …*) | the clause |
| `clause_cloze` | Complete this rule from the *{policy}* (section …): "*first half* …" | the full clause |

Long manuals are capped at 150 pairs per document so they do not dominate. Duplicated instructions are removed.

**Optional — LLM synthetic generation** (`build_instruction_dataset("llm")`, needs `ANTHROPIC_API_KEY` or `OPENAI_API_KEY`). Exact prompt template used per 600-word chunk:
```
Read the text below and generate {n} instruction-response pairs in JSON format based ONLY on this text. Each entry must have instruction and response keys.
The instruction must be a question an employee or HR executive would realistically ask; the response must answer it using only facts stated in the text (quote numbers, limits and conditions exactly). Return a JSON list only, with no commentary.

Document: {title}
---
{chunk}
---
```
**Split:** 80 / 20 by *section group* — all pairs generated from the same section go to the same split, so the eval set measures generalisation to unseen clauses rather than re-phrasings of training clauses.

In [ ]:
from src.instruction_data import build_instruction_dataset
METHOD = "heuristic"   # or "llm" if an API key is configured
inst_stats = build_instruction_dataset(METHOD)
inst = pd.read_json(config.INSTRUCTION_DIR / "instruction_dataset.jsonl", lines=True)
print(f"Train: {inst_stats['train_pairs']} | Eval: {inst_stats['eval_pairs']} | Total: {inst_stats['total_pairs']}")
display(inst.groupby("type").size().to_frame("pairs"))
display(inst.sample(8, random_state=1)[["type", "instruction", "response", "source"]])

**Inferences — B1**
* **Size and split:** ~2 000 grounded pairs from all cleaned documents, split ≈ 80 / 20 by section group (counts printed above). That is enough for a 1.1 B model to learn the *format* “question → answer from the policy” without being so large that QLoRA training becomes expensive.
* **Grounding:** every response is copied from the source policy (no generated text), so the dataset cannot contain hallucinated facts; `source` and `section` give provenance for every pair.
* **Mix:** clause-level pairs (cloze + rule) dominate, which teaches precise, single-rule answers; section-level pairs teach longer summary-style answers.
* **Quality controls added after inspecting the first version:** tables of contents, annexures and blank forms are skipped; dot-leader/form-blank text is rejected; PDF-split words (“exc eeding”) are re-joined in Step 1; duplicate responses are removed; documents get curated titles from `data/sources.csv` (e.g. “AICB HR Manual” instead of “Aicb Hr Manual”).
* **Limitations:** questions are templated (“What does the … require regarding …?”), so the model will be less robust to free-form phrasing; the optional LLM method (template above) would add natural paraphrases at the cost of a factuality check.

## B2 — QLoRA fine-tuning with **Adapter B (Balanced)**  [2 marks]
| Setting | Value |
|---|---|
| Base weights | CPT checkpoint from Step 4, **4-bit NF4**, double quantisation, compute dtype bf16 (L40S/A100) or fp16 (T4) |
| Adapter | **B — r = 16, α = 32, target `q_proj`, `v_proj`**, dropout 0.05 |
| Trainer | TRL `SFTTrainer`, prompt/completion conversations rendered with the chat template; loss on the assistant completion only |
| Optimiser / LR | paged 8-bit AdamW, 2e-4, cosine, 5 % warm-up, 2 epochs, batch 4 × grad-accum 4, max length 512 |

**Why Adapter B?** Adapter A (r = 8) risks under-fitting multi-clause answers; Adapter C (r = 32 + `o_proj`) roughly doubles adapter parameters and VRAM for little gain on a ~10³-pair dataset. B is the quality/cost balance point. α/r = 2 keeps the effective LoRA scale the same as the common r=8/α=16 setting.

**Chat template:** the TinyLlama *base* tokenizer has no chat template, so the official TinyLlama-1.1B-Chat-v1.0 (Zephyr-style) template — `<|system|> … </s> <|user|> … </s> <|assistant|> …` — is attached to the same tokenizer (no vocabulary change: the markers are ordinary text tokens).

In [ ]:
from src.train_qlora import train_qlora, load_chat_tokenizer
chat_tok = load_chat_tokenizer()
print(chat_tok.apply_chat_template([{"role": "system", "content": config.SYSTEM_PROMPT},
                                    {"role": "user", "content": "How many days of notice are required?"},
                                    {"role": "assistant", "content": "60 days."}], tokenize=False))
sft_summary, sft_rows = train_qlora("B")

In [ ]:
log = pd.DataFrame(sft_rows)
fig, ax = plt.subplots(figsize=(8, 3.8))
t = log.dropna(subset=["loss"]); e = log.dropna(subset=["eval_loss"])
ax.plot(t.step, t.loss, label="train loss"); ax.plot(e.step, e.eval_loss, "o-", label="eval loss")
ax.set_xlabel("step"); ax.set_ylabel("loss (completion tokens)"); ax.set_title("QLoRA Adapter B"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(config.FIGURE_DIR / "sft_adapter_B_loss.png", dpi=150); plt.show()
display(pd.Series({k: v for k, v in sft_summary.items() if k != "hyperparameters"}).to_frame("value"))

**Inferences — B2**
* **Parameter efficiency:** Adapter B trains 2,252,800 parameters (22 layers × [16·(2048+2048) for `q_proj` + 16·(2048+256) for `v_proj`]) = **0.20 % of the 1.10 B model**. The printed 0.365 % is relative to 618 M because 4-bit weights are stored two per byte, so `numel()` of the quantised model counts roughly half the real parameters.
* **Memory:** peak GPU memory ≈ 2 GB versus 12 GB for full CPT — the 4-bit base needs ~0.7 GB and only the small LoRA matrices carry gradients and optimiser state. This is what makes QLoRA practical on a T4.
* **Learning curve:** completion-only loss falls from ≈ 1.75 to ≈ 1.45 and mean token accuracy settles at ≈ 64–65 %; held-out loss improves from ≈ 1.62 to ≈ 1.57 and is essentially flat after the first epoch. The small train–eval gap means the adapter learned the answer format and generalises to unseen sections; a second epoch adds little, so 1–1.5 epochs would suffice.
* **Why loss stays ≈ 1.5:** responses are verbatim policy clauses, and an unseen clause cannot be predicted word-for-word — the remaining loss is mostly irreducible content, not a failure to learn the format.

## B3 — Evaluation on the same 3 domain prompts  [1 mark]
The three Step-3/Step-5 topics are asked as questions through the chat template. Columns: raw completions of the base and CPT models (Steps 3/5), the CPT model through the chat template with the adapter **disabled**, and the CPT model **with Adapter B**. Held-out pairs are also scored with ROUGE-L against the reference answers.

In [ ]:
from src.evaluate_sft import evaluate_adapter
domain_cmp, general_cmp, heldout, sft_eval = evaluate_adapter("B")
display(domain_cmp)
display(general_cmp)
display(pd.Series(sft_eval).to_frame("value"))
display(heldout.head(5)[["instruction", "reference", "cpt_no_adapter", "sft_adapter", "rougeL_cpt", "rougeL_sft"]])

In [ ]:
print(f"* Domain probes stating the reference fact — CPT without adapter: {sft_eval['domain_prompts_with_reference_fact']['cpt_no_adapter']}/3, "
      f"with Adapter B: {sft_eval['domain_prompts_with_reference_fact']['sft_adapter']}/3")
print(f"* Held-out ROUGE-L ({sft_eval['heldout_pairs_scored']} pairs): no adapter {sft_eval['rougeL_cpt_no_adapter']:.3f} "
      f"-> Adapter B {sft_eval['rougeL_sft_adapter']:.3f}")
print(f"* General prompts still correct after SFT: {sft_eval['general_prompts_correct_after_sft']}/3")

**Observations — B3** — *read together with the tables above and edit if your run differs.*
* **Format:** without the adapter the CPT model treats the chat prompt as a document to continue (repeats the question, lists further clauses, does not stop); with Adapter B it answers directly in policy wording and ends with `</s>` — the main, reliable effect of instruction tuning.
* **Grounding on the three probes:** the questions name the policy (“Under the MyGov Leave Policy, …”) exactly as the instruction data does, and the MyGov sections are in the SFT training split. Compare the `sft_has_fact` column with the base/CPT columns. Where the adapter misses, it typically answers with a *related* clause from another organisation's manual (e.g. a 90-day leave cap, Rs. 1500/day, 30-day notice). In an earlier run whose questions did *not* name the organisation, all three probes were answered from other manuals — with 30+ organisations' policies in the corpus, a question must say *whose* policy it is about.
* **Held-out ROUGE-L** quantifies the improvement on 30 unseen questions beyond the three hand-picked probes.
* **General knowledge after SFT** (second table): asked through the chat template with the HR-assistant system prompt, the adapter tends to answer *every* question in policy style (an earlier run answered “Water boils at” with “105°C … measured by a thermometer” and the speed of light with a salary figure). LoRA does not change the base weights — the CPT model alone still answered all three correctly in Step 5B — but the adapter has learned to map any question to an HR-policy answer. This is format/instruction over-specialisation, not loss of knowledge; mixing a small share of general instruction data into SFT, or routing non-HR questions to the base model (the optional multi-adapter extension), would fix it.
* **Next steps:** LLM-paraphrased instructions, Adapter C (higher capacity), naming the organisation in each question, or retrieval-augmented prompting (Assignment 2B) to anchor answers to the right policy.

## Summary
| Metric | Result |
|---|---|
| Corpus after cleaning | see Step 1 summary |
| Tokens / packed 2048-blocks | see Step 2 |
| CPT loss (start → end) | see Step 4 |
| Domain PPL base → CPT | see Step 5A |
| Forgetting verdicts | see Step 5B |
| Instruction pairs train / eval | see B1 |
| Adapter B ROUGE-L (no adapter → adapter) | see B3 |

In [ ]:
rows = {
  "documents (raw → clean)": f"{summary['documents_before']} → {summary['documents_after']}",
  "train tokens / blocks": f"{tok_stats['splits']['train']['total_tokens']:,} / {tok_stats['splits']['train']['packed_sequences']}",
  "CPT loss start → end": f"{cpt_summary['start_train_loss']:.3f} → {loss_stats['last_logged_loss']:.3f}",
  "domain PPL base → CPT (held-out 10%)": f"{ppl['base_ppl']:.2f} → {ppl['cpt_ppl']:.2f} (−{ppl['ppl_reduction_percent']}%)",
  "domain PPL base → CPT (unseen documents)": (f"{ppl['unseen_base_ppl']:.2f} → {ppl['unseen_cpt_ppl']:.2f} (−{ppl['unseen_ppl_reduction_percent']}%)"
                                             if "unseen_base_ppl" in ppl else "n/a"),
  "forgetting verdicts": ", ".join(forgetting.verdict),
  "instruction pairs train / eval": f"{inst_stats['train_pairs']} / {inst_stats['eval_pairs']}",
  "ROUGE-L no adapter → Adapter B": f"{sft_eval['rougeL_cpt_no_adapter']:.3f} → {sft_eval['rougeL_sft_adapter']:.3f}",
}
display(pd.Series(rows).to_frame("result"))

## Export
Copies `instruction_dataset.jsonl` to the project root and exports this notebook to HTML.

**Important:** `nbconvert` converts the `.ipynb` *file on disk*, not what you see in the browser. During *Run All* the file on disk only contains the last autosave, so an HTML exported automatically at the end would show a half-finished run. This cell therefore checks the saved file first and only exports when every code cell above has saved output.

Recommended: run the whole notebook headless with `bash run_notebook.sh` from the project root — it executes all cells, saves the outputs into the `.ipynb`, and then writes the HTML, with no manual saving. Interactively: after *Run All* finishes, press **Ctrl+S** (File → Save) and run this cell again.

In [ ]:
import shutil, nbformat
shutil.copy(config.INSTRUCTION_DIR / "instruction_dataset.jsonl", "instruction_dataset.jsonl")
NB = "notebooks/CorpPolicyLM_Assignment1A.ipynb"
saved = nbformat.read(NB, as_version=4)
code_cells = [c for c in saved.cells if c.cell_type == "code"][:-1]   # every code cell except this one
unsaved = sum(c.get("execution_count") is None for c in code_cells)
if os.environ.get("NOTEBOOK_HEADLESS") == "1":
    print("Headless run: run_notebook.sh exports the HTML after all cells have finished and been saved.")
elif unsaved:
    print(f"⚠ The saved notebook is stale: {unsaved} of {len(code_cells)} code cells have no output on disk.\n"
          "  Press Ctrl+S (File → Save) and run this cell again — or use `bash run_notebook.sh`.")
else:
    subprocess.run(["jupyter", "nbconvert", "--to", "html", NB, "--output-dir", "notebooks"], check=True)
    print("Exported:", sorted(f for f in os.listdir("notebooks") if f.endswith(".html")))